In [ ]:
#Osnovne biblioteke

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from matplotlib.ticker import FuncFormatter
from collections import Counter
from scipy.stats import randint
from pathlib import Path


In [ ]:
#Scikit-learn – pipeline i transformacije

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    QuantileTransformer,
    FunctionTransformer,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
#Scikit-learn – modeli

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    RidgeCV,
    Lasso,
    ElasticNet
)
from sklearn.ensemble import (
    RandomForestRegressor,
    StackingRegressor
)
from sklearn.neural_network import MLPRegressor

In [ ]:
#Scikit-learn – evaluacija, metrika i 

from sklearn.model_selection import (
    train_test_split,
    cross_val_predict,
    KFold,
    GridSearchCV,
    RandomizedSearchCV,
    learning_curve
)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_squared_log_error,
    make_scorer
)
from sklearn.inspection import permutation_importance

In [ ]:
#Boosting modeli

import xgboost as xgb
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [ ]:
current_directory = str(Path.cwd())
print(current_directory)

In [ ]:
data = pd.read_csv(current_directory+"/train.csv")
df = pd.DataFrame(data)
pd.set_option('display.max_columns', 300)
pd.set_option('display.max_rows', 300)

In [ ]:
df.head()

In [ ]:
print(df.dtypes.unique())

In [ ]:
df.drop(columns=['id'], inplace=True)

In [ ]:
df.shape

In [ ]:
# prag za periferiju (npr. stanovi dalje od 30 km od Kremlja)
df = df[df["kremlin_km"] <= 30].copy()

In [ ]:
target = df["price_doc"]
print(target.isna().sum())

In [ ]:
print(df["price_doc"].head())
print(df["price_doc"].min(), df["price_doc"].max())

In [ ]:

plt.figure(figsize=(10,6))
plt.boxplot(df["price_doc"], vert=False, patch_artist=True)

plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.title("Boxplot za target (price_doc)")
plt.xlabel("Cena stana (RUB)")
plt.show()

In [ ]:
#df = df.sample(n=10000, random_state=42)

In [ ]:
plt.figure(figsize=(10,6))
plt.boxplot(df["price_doc"], vert=False, patch_artist=True)

plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.title("Boxplot za target (price_doc)")
plt.xlabel("Cena stana (RUB)")
plt.show()

In [ ]:
# originalni target
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(df["price_doc"], kde=True, bins=50, color="blue")
plt.title("Originalna raspodela (price_doc)")
plt.xlabel("Cena stana (RUB)")

# log-transformisani target
plt.subplot(1,2,2)
sns.histplot(np.log1p(df["price_doc"]), kde=True, bins=50, color="green")
plt.title("Log-transformisana raspodela (log(price_doc))")
plt.xlabel("log(cena)")

plt.tight_layout()
plt.show()

In [ ]:
new_cols = []

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
start = df["timestamp"].min()

df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month

df.drop(columns=['timestamp'], inplace=True)
new_cols.extend(["year","month"])

In [ ]:
df.shape

In [ ]:
df["floor_ratio"] = df.apply(
    lambda row: row["floor"] / row["max_floor"] if pd.notnull(row["max_floor"]) and row["max_floor"] > 0 else 0,
    axis=1
)

In [ ]:
df["build_age"] = df["year"] - df["build_year"]
new_cols.extend(["floor_ratio","build_age"])

In [ ]:
df.shape

In [ ]:
cols = [c for c in df.columns if c != 'price_doc'] + ['price_doc']
df = df[cols]

In [ ]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [ ]:
print(y.name)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
y_train.head()

In [ ]:
"""Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

mask = (y_train >= lower_bound) & (y_train <= upper_bound)

X_train = X_train.loc[mask]
y_train = y_train.loc[mask]"""

lower = y_train.quantile(0.01)
upper = y_train.quantile(0.85)

mask = (y_train >= lower) & (y_train <= upper)

X_train_clean = X_train.loc[mask]
y_train_clean = y_train.loc[mask]

In [ ]:
df = df.dropna(subset=["price_doc"]).copy()

In [ ]:
plt.figure(figsize=(10,6))
plt.boxplot(y_train_clean, vert=False, patch_artist=True)

plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.title("Boxplot za target (price_doc) nakon uklanjanja autlajera")
plt.xlabel("Cena stana (RUB)")
plt.show()

In [ ]:
# originalni target
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(y_train_clean, kde=True, bins=50, color="blue")
plt.title("Originalna raspodela (price_doc)")
plt.xlabel("Cena stana (RUB)")

# log-transformisani target
plt.subplot(1,2,2)
sns.histplot(np.log1p(y_train_clean), kde=True, bins=50, color="green")
plt.title("Log-transformisana raspodela (log(price_doc))")
plt.xlabel("log(cena)")

plt.tight_layout()
plt.show()

In [ ]:
df["price_doc_log"] = np.log1p(df["price_doc"])

In [ ]:
y_train_clean = np.log1p(y_train_clean)

### Test

In [ ]:
def missing_values_ratio(df, ratio_for_na_values=0.3, target_col="price_doc", corr_threshold=0.4):
    # 1. Missing ratio for each column
    missing_ratio = df.isnull().mean().sort_values(ascending=False)
    
    # 2. Select only columns with > threshold % missing
    cols_over_n = missing_ratio[missing_ratio > ratio_for_na_values]
    cols_to_drop = []
    
    if not cols_over_n.empty:
        print(f"Kolone sa preko {round(ratio_for_na_values*100)}% nedostajućih vrednosti:\n")
        
        # 3. For each such column, compute correlation with target
        for col in cols_over_n.index:
            if df[col].dtype in ['int64', 'float64']:  # only numeric can correlate
                corr_val = df[[col, target_col]].corr().iloc[0, 1]  # correlation col <-> target
                if (corr_val is None) or (abs(corr_val) < corr_threshold):
                    cols_to_drop.append(col)
                print(f"{col}: {cols_over_n[col]*100:.2f}% NaN, korelacija sa targetom = {corr_val:.3f}")
        return cols_to_drop
    else:
        print("Nema takvih kolona")
        return []

In [ ]:
columns_to_drop = missing_values_ratio(df, 0.3, "price_doc", 0.4)

In [ ]:
print(len(X_test.columns.tolist()))

In [ ]:
df = df.drop(columns = columns_to_drop)

In [ ]:
X_test = X_test.drop(columns=columns_to_drop)  
X_train = X_train_clean.drop(columns=columns_to_drop) 

In [ ]:
X_train.shape, X_test.shape

In [ ]:
train_imputer = SimpleImputer

In [ ]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns

In [ ]:
train_num = X_train[num_cols].copy()
train_cat = X_train[cat_cols].copy()

In [ ]:
train_num.shape, train_cat.shape

In [ ]:
for cat in cat_cols:
    print(cat)
    print(train_cat[cat].unique())

In [ ]:
ordinal_cols = ["ecology"] 
freq_cols = ["sub_area"]
nominal_cols = [col for col in cat_cols if col not in ordinal_cols + freq_cols]

In [ ]:
print(nominal_cols)

In [ ]:
train_cat["ecology"] = train_cat["ecology"].replace("no data", np.nan)
train_cat = train_cat.replace(["nan", "NaN", ""], np.nan)
imputer = SimpleImputer(strategy="most_frequent").set_output(transform="pandas")
train_cat = imputer.fit_transform(train_cat)

In [ ]:
train_cat.isna().sum().sum()

In [ ]:
imputer_num = SimpleImputer(strategy="median").set_output(transform="pandas")
train_num = imputer.fit_transform(train_num)

train_num.isna().sum().sum()

In [ ]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None, normalize=True):
        self.columns = columns
        self.normalize = normalize
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.columns:
            self.freq_maps[col] = X[col].value_counts(normalize=self.normalize)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X

In [ ]:
def categorical_to_numeric(df_obj, nominal_cols, ordinal_cols, freq_cols):
    if freq_cols:
        fe = FrequencyEncoder(columns=freq_cols, normalize=True)
        df_obj[freq_cols] = fe.fit_transform(df_obj[freq_cols])


    if nominal_cols:
        ohe = OneHotEncoder(drop="if_binary", sparse_output=False, handle_unknown="ignore")
        df_encoded = pd.DataFrame(
            ohe.fit_transform(df_obj[nominal_cols]),
            columns=ohe.get_feature_names_out(nominal_cols),
            index=df_obj.index
        )
        df_obj = df_obj.drop(columns=nominal_cols)
        df_obj = pd.concat([df_obj, df_encoded], axis=1)

    if ordinal_cols:
        order = [["excellent", "good", "satisfactory", "poor"]]
        enc = OrdinalEncoder(categories=order)
        for col in ordinal_cols:
            df_obj[col] = enc.fit_transform(df_obj[[col]])

    return df_obj

In [ ]:
train_cat_to_num = categorical_to_numeric(train_cat, nominal_cols, ordinal_cols, freq_cols)

In [ ]:
train_cat_to_num.shape

In [ ]:
train_cat_with_target = train_cat_to_num.copy()
train_cat_with_target['price_doc'] = y_train 

corr = train_cat_with_target.corr()["price_doc"].sort_values(ascending=False)
print(corr[(corr >= 0.25) | (corr <= -0.25)])

In [ ]:
def columns_selector(train_num_with_target, corr_treshold = 0.25):
    
    corr_matrix = train_num_with_target.corr(numeric_only=True)
    target_corr = corr_matrix["price_doc"].drop("price_doc")
    selected_features = target_corr[target_corr.abs() > corr_treshold].index.tolist()
    
    print(f"Feature-i sa korelacijom > {corr_treshold} sa targetom:")
    print(selected_features)

    return selected_features

In [ ]:
corr_treshold = 0.1
selected_features = columns_selector(train_cat_with_target, corr_treshold = corr_treshold)
plt.figure(figsize=(10,8))
sns.heatmap(train_cat_with_target[selected_features].corr(), annot=True, cmap="coolwarm", center=0)
plt.title(f"Korelacija feature-a koji imaju korelaciju > {corr_treshold} sa targetom")
plt.show()

In [ ]:
train_cat_to_num.head()

In [ ]:
def winsorize_all_numeric(df, lower=0.01, upper=0.99):
    df_copy = df.copy()
    num_cols = df_copy.select_dtypes(include=["number"]).columns
    
    for col in num_cols:
        low, high = df_copy[col].quantile([lower, upper])
        df_copy[col] = df_copy[col].clip(lower=low, upper=high)
    
    return df_copy

df = winsorize_all_numeric(df, lower=0.01, upper=0.99)

In [ ]:
df.shape

In [ ]:
train_num_with_target = train_num.copy()
train_num_with_target['price_doc'] = y_train 

corr = train_num_with_target.corr()["price_doc"].sort_values(ascending=False)
print(corr[(corr >= 0.2) | (corr <= -0.2)])

In [ ]:
corr_treshold = 0.2

selected_features_num = columns_selector(train_num_with_target, corr_treshold = corr_treshold)
plt.figure(figsize=(10,8))
sns.heatmap(train_num_with_target[selected_features_num].corr(), annot=True, cmap="coolwarm", center=0)
plt.title(f"Korelacija feature-a koji imaju korelaciju > {corr_treshold} sa targetom")
plt.show()

In [ ]:
def drop_hight_corr_pairs(train_num_with_target, selected_features, pairs_treshold=0.85):

    corr_matrix = train_num_with_target[selected_features].corr()
    
    df_cleared = train_num_with_target[selected_features]
    
    target_corr = df_cleared.corrwith(y_train).abs()
    
    high_corr_pairs = [
        (col1, col2)
        for col1 in corr_matrix.columns
        for col2 in corr_matrix.columns
        if col1 < col2 and corr_matrix.loc[col1, col2] > pairs_treshold
    ]
    
    print(high_corr_pairs)
    print(f"len: {len(high_corr_pairs)}")
    print()
    print("col1 to col2 ratio")
    print()
    cols_to_drop = []
    
    for c1, c2 in high_corr_pairs:
        if target_corr.get(c1, 0) < target_corr.get(c2, 0):
            print(f"{c1} < {c2}  :  {target_corr.get(c1, 0)} < {target_corr.get(c2, 0)}")
            cols_to_drop.append(c2)
        else:
            print(f"{c2} < {c1}  :  {target_corr.get(c2, 0)} < {target_corr.get(c1, 0)}")       
            cols_to_drop.append(c1)
    
    df_cleared = df_cleared.drop(columns=cols_to_drop)
    print()
    print("columns to keep :")
    print(df_cleared.columns)
    print(f"len: {len(df_cleared.columns)}")

    return df_cleared

In [ ]:
df_cleared = drop_hight_corr_pairs(train_num_with_target, selected_features_num, pairs_treshold=0.9)

### 1. Dimenzije i sobe

full_sq → ukupna kvadratura stana (m²).

life_sq → stambena površina stana (m²), dakle samo sobe, bez hodnika, balkona i pomoćnih prostorija.

num_room → broj soba u stanu.

In [ ]:
flags = []

In [ ]:
def bar_plot(df, col, y_label="price_doc"):
    plt.figure(figsize=(6,4))
    plt.hist(df[col].dropna(), bins=40, color="steelblue", edgecolor="black")
    plt.title(f"Distribucija {col}")
    plt.xlabel(col)
    plt.ylabel(y_label)
    plt.show()

def scatter_plot(df, col, target = "price_doc"):
    plt.figure(figsize=(6,4))
    plt.scatter(df[col], df[target], alpha=0.3)
    plt.title(f"{col} vs {target}")
    plt.xlabel(col)
    plt.ylabel(target)
    plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(df["full_sq"], df["life_sq"], alpha=0.5)
plt.xlabel("full_sq")
plt.ylabel("life_sq")
plt.title("Scatterplot: full_sq vs life_sq")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Izračunaj korelaciju sa targetom
corr_full = df["full_sq"].corr(df["price_doc"])
corr_life = df["life_sq"].corr(df["price_doc"])

# Ubaci u dict pa u DataFrame za lakši plot
corr_dict = {
    "full_sq": corr_full,
    "life_sq": corr_life
}

sns.barplot(x=list(corr_dict.keys()), y=list(corr_dict.values()))
plt.title("Korelacija sa targetom (price_doc)")
plt.ylabel("Pearson korelacija")
plt.show()

print("Korelacija full_sq sa targetom:", corr_full)
print("Korelacija life_sq sa targetom:", corr_life)

* Solidna korelacija full_sq i life_sq sa targetom

* Loša međusobna korelacija

* Treba zadržati oba

In [ ]:
bad_full = df[df["full_sq"] <= 0]
bad_life = df[df["life_sq"] <= 0]
bad_ratio = df[df["life_sq"] > df["full_sq"]]
bad_room_zero = df[df["num_room"] == 0]
bad_room_high = df[df["num_room"] > 10]
df["sqm_per_room"] = df["full_sq"] / df["num_room"].replace(0, np.nan)
bad_sqm_low = df[df["sqm_per_room"] < 8]
bad_sqm_high = df[df["sqm_per_room"] > 80]


print("Provera dal postoje stanovi sa kvadraturom < 0")
print("Kvadratura <= 0:", bad_full.shape[0])
print("life_sq <= 0:", bad_life.shape[0])
print("life_sq > full_sq:", bad_ratio.shape[0])
print("num_room = 0:", bad_room_zero.shape[0])
print("num_room > 10:", bad_room_high.shape[0])
print("sqm_per_room < 8:", bad_sqm_low.shape[0])
print("sqm_per_room > 80:", bad_sqm_high.shape[0])

In [ ]:
# --- life_sq ---
df["life_sq_missing_or_zero"] = (df["life_sq"] <= 0).astype(int)
df["life_sq_bigger_full"] = (df["life_sq"] > df["full_sq"]).astype(int)


df.loc[df["life_sq"] <= 0, "life_sq"] = np.nan
df.loc[df["life_sq"] > df["full_sq"], "life_sq"] = np.nan

# --- num_room ---
df["num_room_zero"] = (df["num_room"] == 0).astype(int)
df["num_room_outlier"] = (df["num_room"] > 10).astype(int)

df.loc[df["num_room"] == 0, "num_room"] = np.nan
df.loc[df["num_room"] > 10, "num_room"] = np.nan

df["sqm_per_room"] = df["full_sq"] / df["num_room"].replace(0, np.nan)

df["sqm_too_small"] = (df["sqm_per_room"] < 8).astype(int)
df["sqm_too_big"] = (df["sqm_per_room"] > 80).astype(int)

df.loc[df["sqm_per_room"] < 8, "num_room"] = np.nan
df.loc[df["sqm_per_room"] > 80, "num_room"] = np.nan


flags.extend(["life_sq_bigger_full","life_sq_bigger_full","num_room_zero","num_room_outlier","sqm_too_small","sqm_too_big"])
new_cols.extend(["sqm_per_room"])

Poredjenje full_sq sa num_room

In [ ]:
scatter_plot(df, col = "num_room",target="full_sq")

### 2. Saobraćaj i udaljenosti do centra

* metro_min_avto – vreme u minutima do najbliže metro stanice kolima

* kremlin_km – udaljenost u kilometrima do Kremlja

In [ ]:
cols = ["metro_min_avto", "kremlin_km"]

print(df[cols].describe())

In [ ]:
cols = ["metro_min_avto", "kremlin_km"]

thresholds = [0, 1, 2, 5, 10, 20, 50, 100]

for col in cols:
    print(f"\n=== {col} ===")
    for t in thresholds:
        count = (df[col] <= t).sum()
        print(f" <= {t}: {count}")


In [ ]:
mask = (df["metro_min_avto"] < 0.5) | (df["kremlin_km"] < 0.1)
df.loc[mask, ["metro_min_avto", "kremlin_km"]] = np.nan
df["is_near_kremlin"] = (df["kremlin_km"] < 1).astype(int)
flags.extend(["is_near_kremlin"])

In [ ]:
df.loc[mask, ["metro_min_avto", "kremlin_km"]]

In [ ]:
bar_plot(df, col = "metro_min_avto")

In [ ]:
df["metro_far"] = (df["metro_min_avto"] > 30).astype(int)
df.loc[df["metro_min_avto"] > 30, "metro_min_avto"] = np.nan
flags.extend(["metro_far"])

In [ ]:
scatter_plot(df, col = "metro_min_avto")

In [ ]:
bar_plot(df, col = "kremlin_km")

In [ ]:
scatter_plot(df, col = "kremlin_km")

In [ ]:
plt.scatter(df["kremlin_km"], df["metro_min_avto"], alpha=0.3)
plt.xlabel("Udaljenost od Kremlja (km)")
plt.ylabel("Vreme do metroa kolima (min)")
plt.title("Provera: udaljenost od centra vs vreme do metroa")
plt.show()

### 3. Okruženje i infrastruktura

* workplaces_km – udaljenost u kilometrima do poslovnih centara

* catering_km – udaljenost u kilometrima do najbližeg restorana ili ketering objekta

* big_church_km – udaljenost u kilometrima do najbliže velike crkve

In [ ]:
cols = ["workplaces_km", "catering_km", "big_church_km"]

# pragovi za "previše daleko"
thresholds = {
    "workplaces_km": 50,   # poslovni centri su uglavnom u centru
    "catering_km": 20,     # restorani teško da su preko 20 km daleko
    "big_church_km": 30    # velike crkve obično u gradu
}

for col in cols:
    print(f"\n=== {col} ===")
    print(f"Ukupno vrednosti: {df[col].shape[0]}")
    print(f"=0: {(df[col] == 0).sum()}")
    print(f"> {thresholds[col]} km: {(df[col] > thresholds[col]).sum()}")
    print(f"Min: {df[col].min()}, Max: {df[col].max()}")
    print()

# dodatne neusaglašenosti sa kremlin_km
print("\n=== Neusaglašenosti u odnosu na kremlin_km ===")
print("workplaces dalje od 2x kremlin:", (df["workplaces_km"] > 2 * df["kremlin_km"]).sum())
print("catering dalje od 10 km i kremlin < 5 km:", ((df["catering_km"] > 10) & (df["kremlin_km"] < 5)).sum())
print("big_church dalje od 20 km i kremlin < 5 km:", ((df["big_church_km"] > 20) & (df["kremlin_km"] < 5)).sum())


In [ ]:
df["workplaces_zero"] = (df["workplaces_km"] == 0).astype(int)
df["workplaces_far"] = (df["workplaces_km"] > 50).astype(int)
df["big_church_far"] = (df["big_church_km"] > 30).astype(int)

df.loc[df["workplaces_km"] == 0, "workplaces_km"] = np.nan
df.loc[df["workplaces_km"] > 50, "workplaces_km"] = np.nan
df.loc[df["big_church_km"] > 30, "big_church_km"] = np.nan

flags.extend(["workplaces_zero","workplaces_far","big_church_far"])

### 4. Posebne lokacije

* nuclear_reactor_km – udaljenost do nuklearnog reaktora

* detention_facility_km – udaljenost do zatvora / pritvorske jedinice


In [ ]:
cols = {
    "nuclear_reactor_km": 100,       # prag za ekstremno daleko
    "detention_facility_km": 60      # prag za ekstremno daleko
}

for col, thr in cols.items():
    print(f"\n=== {col} ===")
    print(f"Ukupno vrednosti: {df[col].shape[0]}")
    print(f"=0: {(df[col] == 0).sum()}")
    print(f"<0.1 km: {(df[col] < 0.1).sum()}")   # praktično "u objektu"
    print(f"> {thr} km: {(df[col] > thr).sum()}")
    print(f"Min: {df[col].min()}, Max: {df[col].max()}")


In [ ]:
df["detention_far"] = (df["detention_facility_km"] > 60).astype(int)
df.loc[df["detention_facility_km"] > 60, "detention_facility_km"] = np.nan
flags.extend(["detention_far"])

In [ ]:
df["price_per_m2"] = df["price_doc"] / df["full_sq"]
df = df[df["price_per_m2"].notnull() & (df["price_per_m2"] < 1_000_000)].copy() # ukloni outliere

# Log transformacija
df["log_price_per_m2"] = np.log(df["price_per_m2"])

In [ ]:
print(df.shape)

In [ ]:
df_num = df.select_dtypes("number")

In [ ]:
corr_treshold = 0.3
selected_features = columns_selector(df_num, corr_treshold = corr_treshold)
plt.figure(figsize=(10,8))
sns.heatmap(df_num[selected_features].corr(), annot=True, cmap="coolwarm", center=0)
plt.title(f"Korelacija feature-a koji imaju korelaciju > {corr_treshold} sa targetom")
plt.show()

In [ ]:
print(flags)

In [ ]:
print(new_cols)

In [ ]:
df_cleared = drop_hight_corr_pairs(df_num, selected_features, pairs_treshold=0.9)

In [ ]:
hight_corr_cols = ['full_sq', 'life_sq', 'num_room', 'zd_vokzaly_avto_km',
       'sport_count_2000', 'office_sqm_5000', 'trc_count_5000',
       'price_doc_log', 'log_price_per_m2'],

df_cat = df.select_dtypes("object")
cat_cols = df_cat.columns

In [ ]:
df.to_csv("data_ceaned.csv", index=False, encoding="utf-8")